In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, ttest_rel
from itertools import product as iproduct

plt.style.use('seaborn-v0_8-whitegrid')

FAST_TRACK = True
BASELINES  = ['resnet', 'densenet', 'mlp', 'vgg']
COLOURS    = {
    'rgnet':    '#2166ac',
    'resnet':   '#d6604d',
    'densenet': '#f4a582',
    'mlp':      '#92c5de',
    'vgg':      '#b2abd2',
}

In [ ]:
if FAST_TRACK:
    rng   = np.random.default_rng(0)
    MEANS = {
        'rgnet':    {'iid': 0.823, 'hier': 0.791},
        'resnet':   {'iid': 0.819, 'hier': 0.734},
        'densenet': {'iid': 0.821, 'hier': 0.748},
        'mlp':      {'iid': 0.802, 'hier': 0.681},
        'vgg':      {'iid': 0.818, 'hier': 0.722},
    }
    records = {
        arch: {
            ds: (rng.standard_normal(5) * 0.012 + mu).tolist()
            for ds, mu in means.items()
        }
        for arch, means in MEANS.items()
    }
    print('Mode: FAST-TRACK (simulated from Table 1 expected values)')
else:
    import json
    with open('results/h3/h3_results.json') as fh:
        raw = json.load(fh)
    records = {arch: data['accuracies'] for arch, data in raw['records'].items()}
    print('Mode: FULL')

In [ ]:
def cohens_d(a, b):
    diff   = a - b
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2.0)
    return float(diff.mean() / (pooled + 1e-12))

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

comparison = {}
print(f'{"Baseline":<10} {"Dataset":<8} {"RGNet":>7} {"Base":>7} '
      f'{"Adv":>7} {"p":>9} {"Sig":>5} {"d":>7}')
print('-' * 62)

for baseline, dataset in iproduct(BASELINES, ['iid', 'hier']):
    rg  = np.array(records['rgnet'][dataset])
    bs  = np.array(records[baseline][dataset])
    try:
        _, p = wilcoxon(rg, bs, alternative='greater')
    except ValueError:
        _, p = ttest_rel(rg, bs)
    d   = cohens_d(rg, bs)
    adv = rg.mean() - bs.mean()
    comparison[(baseline, dataset)] = {'p_value': p, 'cohens_d': d, 'advantage': adv}
    print(f'{baseline:<10} {dataset:<8} {rg.mean():>7.4f} {bs.mean():>7.4f} '
          f'{adv:>+7.4f} {p:>9.4f} {stars(p):>5} {d:>7.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, dataset in zip(axes, ['iid', 'hier']):
    archs = ['rgnet'] + BASELINES
    for i, arch in enumerate(archs):
        ax.boxplot(
            records[arch][dataset],
            positions=[i], widths=0.5,
            patch_artist=True, notch=True,
            boxprops=dict(facecolor=COLOURS[arch], alpha=0.7),
            medianprops=dict(color='black', linewidth=2),
            flierprops=dict(marker='o', markersize=4),
        )
    ax.set_xticks(range(len(archs)))
    ax.set_xticklabels([a.upper() for a in archs], fontsize=10)
    ax.set_ylabel('Accuracy')
    ax.set_title('IID' if dataset == 'iid' else 'Hierarchical')

    for i, baseline in enumerate(BASELINES, start=1):
        p   = comparison[(baseline, dataset)]['p_value']
        sig = stars(p)
        if sig != 'n.s.':
            y = max(max(records['rgnet'][dataset]),
                    max(records[baseline][dataset])) + 0.005
            ax.annotate(sig, xy=((0 + i) / 2, y), ha='center', fontsize=10)
            ax.plot([0, i], [y - 0.002, y - 0.002],
                    color='#888888', linewidth=0.8)

plt.suptitle('H3: RG-Net vs Baselines  (* p<0.05  ** p<0.01  *** p<0.001)',
             fontsize=12)
plt.tight_layout()
plt.savefig('nb04_h3_comparison.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
hier_ok = all(
    comparison[(b, 'hier')]['p_value'] < 0.01
    and comparison[(b, 'hier')]['cohens_d'] > 0.5
    for b in BASELINES
)
iid_ok = all(
    comparison[(b, 'iid')]['advantage'] >= 0.0
    or comparison[(b, 'iid')]['p_value'] > 0.05
    for b in BASELINES
)
print(f'Hierarchical (p<0.01, d>0.5):  {hier_ok}')
print(f'IID parity maintained:          {iid_ok}')
print(f'H3 VALIDATED:                   {hier_ok and iid_ok}')